# Phase 2 — document-level role resolver (post-processing on sub-9 predictions)

**Settings:** Accelerator **None**, Internet **on**, `HF_TOKEN` secret ticked, and **upload `sub9.json`** as an input.

Cells 1–4 are from my `phase2-lner` notebook (data + metric). **Cell 5 is mine to write** (the resolver). Cell 6 checks it against the fold-1/2 gold; cell 7 applies it to sub 9 and writes the submission zip.

In [ ]:
import json, os, glob, collections
from kaggle_secrets import UserSecretsClient
from huggingface_hub import hf_hub_download

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
REPO = "Exploration-Lab/CS-781-Capstone"
files = ["Train/fold1-text.json", "Train/fold1-gold.json",
         "Train/fold2-text.json", "Train/fold2-gold.json",
         "Train/labels.txt", "Test/fold3-text.json"]
P = {}
for f in files:
    P[f] = hf_hub_download(REPO, f"Phase_2_LNER/{f}", repo_type="dataset", token=HF_TOKEN)
    print(f, "->", os.path.getsize(P[f]) // 1024, "KB")

In [ ]:
t1 = json.load(open(P["Train/fold1-text.json"]))
g1 = json.load(open(P["Train/fold1-gold.json"]))
labels = open(P["Train/labels.txt"]).read().split()

print("labels:", labels)
print("text type:", type(t1).__name__, "| gold type:", type(g1).__name__)
k = list(t1)[0] if isinstance(t1, dict) else 0
print("first id:", k)
print("text sample:", repr(str(t1[k])[:400]))
print("gold sample:", json.dumps(g1[k] if isinstance(g1, dict) else g1[0], ensure_ascii=False)[:800])

In [ ]:
def load(fold):
    t = json.load(open(P[f"Train/{fold}-text.json"]))
    g = json.load(open(P[f"Train/{fold}-gold.json"]))
    return t, g

for fold in ("fold1", "fold2"):
    t, g = load(fold)
    lens = [len(v) for v in t.values()]
    print(f"{fold}: {len(t)} docs | chars per doc: min {min(lens)}, median {sorted(lens)[len(lens)//2]}, max {max(lens)}")

t3 = json.load(open(P["Test/fold3-text.json"]))
lens = [len(v) for v in t3.values()]
print(f"fold3 (test): {len(t3)} docs | chars: min {min(lens)}, median {sorted(lens)[len(lens)//2]}, max {max(lens)}")

In [ ]:
import re, bisect, random

def strict_macro_f1(gold, pred):
    tp, fp, fn = collections.Counter(), collections.Counter(), collections.Counter()
    for d in gold:
        G = {(s, e, l) for s, e, l, *_ in gold[d]}
        Pd = {(s, e, l) for s, e, l, *_ in pred.get(d, [])}
        for x in Pd: (tp if x in G else fp)[x[2]] += 1
        for x in G - Pd: fn[x[2]] += 1
    per = {}
    for l in labels:
        p = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        r = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        per[l] = 2 * p * r / (p + r) if p + r else 0.0
    return sum(per.values()) / len(labels), per

In [ ]:
# ===== MY role resolver (post-processing; only APP/RESP and A.COUNSEL/R.COUNSEL labels change) =====
PARTY = ("APP", "RESP"); COUNSEL = ("A.COUNSEL", "R.COUNSEL"); ROLE4 = PARTY + COUNSEL

def toks(s):
    """Distinctive lowercase name tokens (3+ letters), without generic words (state, union, india, ltd, ors, mr, shri, ...)."""
    raise NotImplementedError

def header_bounds(text):
    """Return (header_end, first_versus_position_or_None).
    header ends at the first 'J U D G M E N T' / 'JUDGMENT' / 'ORDER' / 'BENCH:' / 'CORAM' (else ~1500 chars)."""
    raise NotImplementedError

def header_side(text, s, e, end, div):
    """'APP' / 'RESP' / None for a party span inside the header. Check in this order:
    1. 'X Vs. Y' adjacency: Vs./v./Versus right AFTER the span -> APP; right BEFORE -> RESP
    2. suffix word within ~40 chars after: ..Appellant(s)/Petitioner(s) -> APP, ..Respondent(s) -> RESP
       (NOT if followed by ':' -- 'PETITIONER: X RESPONDENT: Y' means that word labels the NEXT item)
    3. nearest prefix label before: 'PETITIONER:'/'APPELLANT:' -> APP, 'RESPONDENT:' -> RESP
    4. else before/after the first Versus of the header"""
    raise NotImplementedError

def counsel_side(text, s, e):
    """'A' / 'R' / None. Check in this order:
    1. right after the name (stop at the next capitalised 'For ...' label):
       'appearing for / counsel for / for / on behalf of / representing (the) <word>'
    2. nearest capitalised label before the name: 'For the Appellant:', 'Counsel for Respondent :-', 'Agent for ...'
    3. 'The respondent is represented by Mr. X'
    words -> A: appellant, petitioner, applicant, plaintiff, complainant
             R: respondent, defendant, opposite party, state, union, anything starting 'non-'"""
    raise NotImplementedError

def resolve(text, spans):
    """1. party map: token -> side, from header party spans (header_side)
       2. counsel map: token -> majority side over all counsel spans (counsel_side)
       3. relabel every APP/RESP (resp. counsel) span whose tokens point to EXACTLY one side; nothing else changes"""
    raise NotImplementedError


In [ ]:
# ===== check my resolver against GOLD (folds 1+2): flip some role labels, resolve, measure =====
T, G = {}, {}
for f in ("fold1", "fold2"):
    t, g = load(f); T.update(t); G.update(g)
FLIP = {"APP": "RESP", "RESP": "APP", "A.COUNSEL": "R.COUNSEL", "R.COUNSEL": "A.COUNSEL"}
for p in (0.0, 0.5):
    rnd = random.Random(0); st = collections.Counter()
    for d, g in G.items():
        gold = [(s, e, l) for s, e, l, *_ in g]
        noisy = [[s, e, FLIP[l] if (l in FLIP and rnd.random() < p) else l] for s, e, l in gold]
        fixed = resolve(T[d], noisy)
        for (s, e, lg), (_, _, ln), (_, _, lf) in zip(gold, noisy, fixed):
            if lg in ROLE4:
                st[(lg, "b")] += ln == lg; st[(lg, "a")] += lf == lg; st[(lg, "n")] += 1
    print(f"--- {int(p*100)}% of role labels flipped")
    for l in ROLE4:
        n = st[(l, "n")]; print(f"  {l:10s} n={n:4d} | role accuracy {st[(l,'b')]/n:.3f} -> {st[(l,'a')]/n:.3f}")


In [ ]:
# ===== apply my resolver to sub 9 and write the submission zip =====
import zipfile
sub9 = json.load(open(glob.glob("/kaggle/input/**/sub9.json", recursive=True)[0]))
t3 = json.load(open(P["Test/fold3-text.json"]))
out, chg = {}, collections.Counter()
for d, sp in sub9.items():
    out[d] = resolve(t3[d], sp) if sp else sp
    for a, b in zip(sp, out[d]):
        if a[2] != b[2]: chg[(a[2], b[2])] += 1
json.dump(out, open("/kaggle/working/prediction.json", "w"))
with zipfile.ZipFile("/kaggle/working/submission_myroles.zip", "w") as z:
    z.write("/kaggle/working/prediction.json", arcname="prediction.json")
print("docs", len(out), "| entities", sum(len(v) for v in out.values()), "| relabelled", dict(chg))
